**Nessa versão ajustada do codigo original publicada no artigo:**
- são obtidos os mesmos resultados da versão original
- são realizadas mudanças na estrutura do notebook para facilitar a navegabilidade pelo codigo
- serão incluidos os codigos para reprodução de graficos e tabelas contidas no texto da dissertação




# Required Libraries


In [1]:

import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_predict, KFold
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Read data

In [2]:
# Load Dataset (replace 'file_path' with your file location in Colab)
file_path = './content/dados_jamovi.xlsx'  # Upload your dataset to Colab
data = pd.read_excel(file_path)


# Feature Selection 

In [3]:
# Feature Selection 
target_column = 'Velocity (mean) [µm/s]'
features_columns = [col for col in data.columns if col != target_column]

In [4]:
# Prepare Data
X = data[features_columns]
y = data[target_column]

# Experiments

## Traditional Gradient Boosting Regressor 


### Feature engineering

In [5]:
# Copy data
X_run1 = X.copy()
y_run1 = y.copy()

In [6]:
# Impute Missing Values
imputer = SimpleImputer(strategy='mean')
X_imputed = imputer.fit_transform(X_run1)

### Traning model

In [7]:

# Define the Gradient Boosting Regressor
traditional_gbr_model = GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, random_state=42)


### Validation

In [8]:

# Perform 5-Fold Cross-Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_predictions = cross_val_predict(traditional_gbr_model, X_imputed, y_run1, cv=kf)


In [ ]:

# Calculate Metrics for Cross-Validation
cv_mae = mean_absolute_error(y_run1, cv_predictions)
cv_rmse = np.sqrt(mean_squared_error(y_run1, cv_predictions))
cv_r2 = r2_score(y_run1, cv_predictions)


In [ ]:

# Results
print("5-Fold Cross-Validation Metrics:")
print(f"MAE:  {cv_mae}")
print(f"RMSE: {cv_rmse}")
print(f"R^2:  {cv_r2}")


5-Fold Cross-Validation Metrics:
MAE: 234.31456825890695
RMSE: 397.48962627853695
R^2: 0.8868770918785022


In [26]:
results = {
    "model_name": "GBR Tradicional",
    "MAE": cv_mae, 
    "RMSE": cv_rmse, 
    "R2": cv_r2
}

gbr_results = pd.DataFrame(results, index=[0])
gbr_results


,model_name,MAE,RMSE,R2
0,GBR Tradicional,136.04913,261.552951,0.95102



## Physics-Informed Gradient Boosting Regressor


### Feature Engineering

In [11]:
# Copy data
X_run2 = X.copy()
y_run2 = y.copy()
data_run2 = data.copy()

In [12]:
# Physical Constants for Feature Engineering
g = 9.81  # Gravitational acceleration (m/s²)
rho_p = 2600  # Particle density (kg/m³)
rho_f = 1000  # Fluid density (kg/m³)
mu = 0.001  # Dynamic viscosity of water (Pa·s)

# Physics-Informed Feature Engineering
data_run2['Radius (m)'] = data_run2['Radius (max) (mean) [µm]'] * 1e-6  # Convert µm to meters
data_run2['Area (m²)'] = data_run2['Area (mean) [µm²]'] * 1e-12  # Convert µm² to m²

# Correct Reynolds Number
data_run2['Re'] = (rho_f * data_run2[target_column] * 1e-6 * data_run2['Radius (m)']) / mu  # Re = (rho * v * r) / mu

# Adjust Drag Coefficient for Low Reynolds Number (Stokes regime)
data_run2['Cd'] = 24 / data_run2['Re']  # Cd = 24/Re for laminar regime

# Correct Drag Force using the adjusted Cd
data_run2['Drag Force'] = 0.5 * rho_f * data_run2['Cd'] * data_run2['Area (m²)'] * ((data_run2[target_column] * 1e-6) ** 2)  # Fd = 0.5 * rho * Cd * A * v^2

# Stokes' Settling Velocity (para referência)
data_run2['Stokes Velocity'] = (2 / 9) * ((rho_p - rho_f) * g * (data_run2['Radius (m)'] ** 2) / mu)

# Include Physics-Informed Features
physics_features = ['Radius (m)', 'Re', 'Cd', 'Drag Force']
X_piml = pd.concat([X_run2, data_run2[physics_features]], axis=1)


In [13]:

# Impute Missing Values
imputer = SimpleImputer(strategy='mean')
X_piml_imputed = imputer.fit_transform(X_piml)


### Training model

In [14]:

# Define the Gradient Boosting Regressor
piml_gbr_model = GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, random_state=42)


### Validation

In [ ]:

# Perform 5-Fold Cross-Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_predictions = cross_val_predict(piml_gbr_model, X_piml_imputed, y_run2, cv=kf)

# Calculate Metrics for Cross-Validation
pigbr_cv_mae = mean_absolute_error(y_run2, cv_predictions)
pigbr_cv_rmse = np.sqrt(mean_squared_error(y_run2, cv_predictions))
pigbr_cv_r2 = r2_score(y_run2, cv_predictions)

# Results
print("5-Fold Cross-Validation Metrics:")
print(f"MAE: {pigbr_cv_mae}")
print(f"RMSE: {pigbr_cv_rmse}")
print(f"R^2: {pigbr_cv_r2}")


5-Fold Cross-Validation Metrics:
MAE: 136.0491303563949
RMSE: 261.5529512797582
R^2: 0.9510200640158917


In [27]:
results = {
    "model_name": "Physics-Informed GBR",
    "MAE": cv_mae, 
    "RMSE": cv_rmse, 
    "R2": cv_r2
}

pi_gbr_results = pd.DataFrame(results, index=[0])
pi_gbr_results

,model_name,MAE,RMSE,R2
0,Physics-Informed GBR,136.04913,261.552951,0.95102


## Comparing model's performance

In [28]:
all_results = pd.concat([gbr_results, pi_gbr_results])

In [29]:
all_results

,model_name,MAE,RMSE,R2
0,GBR Tradicional,136.04913,261.552951,0.95102
0,Physics-Informed GBR,136.04913,261.552951,0.95102
